# SMILES-VAE — long training run on Colab (A100)

Trains the SMILES variational autoencoder on a GPU runtime, on the Murcko-scaffold-balanced 50k subset of the 305k (no single scaffold/acyclic class dominates, so the VAE learns general chemistry instead of collapsing to a fatty-chain prior).

**Status (2026-07-29):** the 1-layer model is converged at `scale_ll=0.5` (~30 epochs) — reconstruction Tanimoto ~0.80–0.82 across general chemistry, empirical-latent generation ~18–20% and genuinely diverse (82 distinct scaffolds, mean pairwise Tanimoto 0.106), standard `N(0,I)` sampling ~2–4.5% (architecturally capped: the decoder never trains on prior samples). The `LAYERS=2` decoder-capacity test **collapsed** at every `scale_ll` tried (0.5 / 0.3 / 0.1 / even 0.06 mid-ramp) — a powerful decoder ignores `z` and predicts the token marginal (val_acc stuck ~0.24), and no config flag fixed it.

**This run — free-bits posterior-collapse fix:** `LAYERS=2` + `free_bits=0.5`, a per-dimension KL floor (Kingma et al. 2016) that blocks gradient on any latent dim whose KL is below the floor, so `z` must retain ≥0.5 nats/dim and the decoder can't ignore it. Single-variable test: the **exact** config that collapsed (`LAYERS=2`, `scale_ll=0.5`, `wd=0.5`) + `free_bits=0.5` → does it train instead of collapsing, and does the bigger decoder push empirical/std validity above the 1-layer ceiling (~20% / ~2-4.5%)?

On an A100 the full 50k / 30-epoch run finishes in well under an hour.

The notebook clones [`MauricioCafiero/SMILES_VAE`](https://github.com/MauricioCafiero/SMILES_VAE), installs deps, runs `code/run_train.py`, and shows the reconstruction / generation results.

**Run on a GPU runtime:** *Runtime → Change runtime type → GPU (A100).*

In [ ]:
!nvidia-smi

## 1. Clone the repo & install deps

In [ ]:
import os

BRANCH  = "main"
REPO_URL = "https://github.com/MauricioCafiero/SMILES_VAE.git"
ROOT    = "/content"                              # Colab working root — always start here
REPO_DIR = os.path.join(ROOT, "SMILES_VAE")

# ALWAYS cd to the Colab root before checking for the repo. If this cell runs
# while already inside SMILES_VAE (cwd persisted from a previous cell), a
# relative `os.path.isdir("SMILES_VAE")` would be False and we'd clone a NESTED
# copy at SMILES_VAE/SMILES_VAE — the source of several past issues (outputs
# ended up under .../SMILES_VAE/SMILES_VAE/outputs/...). Starting from /content
# makes the check absolute and the clone idempotent.
%cd {ROOT}

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL}
else:
    print(f"Reusing existing clone at {REPO_DIR}")

%cd {REPO_DIR}
!git checkout {BRANCH}
!git pull

# Detect (do NOT auto-delete) a stale nested clone from a past re-run inside the
# repo, so you know it's there. Remove it with:  !rm -rf '{nested}'
nested = os.path.join(REPO_DIR, "SMILES_VAE")
if os.path.isdir(nested):
    print(f"WARNING: stale nested clone at {nested} — remove it with:  !rm -rf '{nested}'")

# rdkit only — smiles_vae.py no longer imports transformers.
# TF, numpy, pandas, scikit-learn, matplotlib, Pillow are preinstalled on Colab.
!pip install -q rdkit

## 2. Configure & train

**Config cell** (next): defines the dataset + hyperparameters used by the training and `--load` cells. Run it before either. It does **not** train.

**Train cell** (after that): runs `code/run_train.py`. Skip it if you only want to re-run generation from saved weights — instead run the config cell, the inspect-results cell (to set `run_dir`), then the `--load` re-generation cell.

Current defaults = the **free-bits posterior-collapse fix**: scaffold-balanced 50k, `LAYERS=2`, `scale_ll=0.5`, `word_dropout_keep=0.5`, `free_bits=0.5`, 30 epochs — the exact config that collapsed without free-bits, now with a per-dim KL floor so the decoder can't ignore `z`. If it trains (val_acc climbs off ~0.24) and beats the 1-layer baseline on empirical/std validity, the bigger decoder was the lever. If recon degrades, the floor is too aggressive for `latent=256` → lower `FREE_BITS` to 0.25; if it still collapses, raise to 1.0. Set `NROWS>0` to subsample, `DATA` to point at another one-column SMILES CSV.

**When you later scale the dataset:** don't point `DATA` at a raw CSV — rebuild a scaffold-balanced set with

```
python code/build_scaffold_dataset.py \
    --input <big_pool.csv> --output data/<name>.csv \
    --target-n <N> --max-tokens 100 --max-per-scaffold 20
```

else the lipid/token-mass class collapse returns. Then re-tune only the data-dependent hparams (`scale_ll` likely lower — more data naturally regularizes the posterior; `latent`/`units`/`layers` likely higher; `free_bits` may lower once the floor has done its job).

In [ ]:
# Config cell — defines the names the !python calls below interpolate
# ({DATA}, {NROWS}, {TEMPERATURE}, ...). Run this BEFORE training (next cell)
# AND before the --load re-generation cell (cell 11) so the names exist in the
# notebook namespace. This cell does NOT train.
DATA = "data/ZN305K_scaffold_balanced_50k.csv"   # one-column SMILES CSV
NROWS    = 0       # 0 = full dataset (50k). Set NROWS>0 to subsample for a quick test.
EPOCHS   = 30      # converged by ~30 (50ep only overfit); matches the 1-layer baseline
GENERATE = 200
SCALE_LL = 0.5     # the config that COLLAPSED at LAYERS=2 without free-bits.
LATENT   = 256
EMB      = 512
UNITS    = 256
LAYERS   = 2       # the decoder that collapses without the free-bits floor.
# Autoregressive-decoder anti-collapse knobs (Bowman 2016):
WORD_DROPOUT_KEEP = 0.5   # 1.0 = off; lower to force latent z usage (0.5 settled)
ANNEAL_EPOCHS     = 10    # ramp scale_ll 0 -> target over this many epochs (0 = off)
# Free-bits per-dim KL floor (Kingma 2016): the POSTERIOR-COLLAPSE FIX for
# LAYERS=2. Blocks gradient on any latent dim whose KL is below the floor, so z
# must keep >= FREE_BITS nats per dim and the decoder can't ignore it. This run
# is the single-variable test: same config that collapsed (LAYERS=2/scale_ll=0.5/
# wd0.5) + free_bits=0.5 -> does it train instead of collapsing to the marginal?
# If recon degrades (floor too aggressive for latent=256), lower to 0.25; if it
# still collapses, raise to 1.0. 0.0 = off (standard VAE, reproduces the collapse).
FREE_BITS = 0.5
# Decode temperature: 0.0 = greedy argmax (deterministic, mode-collapses); 0.7-1.0 = diverse sampling.
# Temp sweep showed temperature is NOT the lever (decoder confident at every token).
# For a post-training temp sweep, re-run only the --load re-gen cell with TEMPERATURE = 0.0 / 0.3 / 0.5.
TEMPERATURE = 0.7
CLIPNORM = 5.0   # Adam gradient-norm clip. 1.0 over-clips LAYERS>=2 (starves
                 # effective LR -> stalls training). 5.0 leaves 1-layer untouched.

# On GPU, cuDNN GRU + strict op-determinism can raise at fit time, so disable it.
%env SMILES_VAE_DISABLE_OP_DETERMINISM=1
!mkdir -p outputs

In [ ]:
# Train. SKIP this cell if you only want to re-run generation from saved
# weights (after a restart: run the config cell above + the inspect-results
# cell to set run_dir, then the --load re-generation cell).
!python code/run_train.py \
    --data {DATA} \
    --nrows {NROWS} --epochs {EPOCHS} --generate {GENERATE} \
    --scale_ll {SCALE_LL} --latent {LATENT} \
    --emb {EMB} --units {UNITS} --layers {LAYERS} \
    --word_dropout_keep {WORD_DROPOUT_KEEP} --anneal_epochs {ANNEAL_EPOCHS} \
    --free_bits {FREE_BITS} \
    --temperature {TEMPERATURE} --clipnorm {CLIPNORM} \
    2>&1 | tee outputs/long_run.log

## 3. Inspect the results

Prints the per-epoch history and the generated SMILES (standard `N(0,I)` vs empirical-latent sampling).

In [ ]:
import os, glob

runs = sorted(glob.glob('outputs/run_*'))
assert runs, 'no run dir found — did training finish?'
run_dir = runs[-1]
print('Latest run:', run_dir, '\n')

print('--- history.csv ---')
print(open(os.path.join(run_dir, 'history.csv')).read())

print('--- generated (empirical) ---')
print(open(os.path.join(run_dir, 'generated_smiles_empirical.txt')).read())

print('--- generated (standard N(0,I)) ---')
print(open(os.path.join(run_dir, 'generated_smiles_standard.txt')).read())

## 4. Reconstruction & generation grids

In [ ]:
from IPython.display import Image, display

for f in ['reconstruction_grid.png', 'generated_grid.png']:
    p = os.path.join(run_dir, f)
    if os.path.exists(p):
        print(p)
        display(Image(p))

## 5. Re-run generation without retraining

Uses `--load` to reuse the saved weights for a larger generation pass. No training happens — useful for sampling many more molecules once the model is trained.

In [ ]:
GEN_N = 500     # how many molecules to generate this pass

!python code/run_train.py --load {run_dir} \
    --data {DATA} \
    --nrows {NROWS} --generate {GEN_N} \
    --scale_ll {SCALE_LL} --latent {LATENT} \
    --emb {EMB} --units {UNITS} --layers {LAYERS} \
    --word_dropout_keep {WORD_DROPOUT_KEEP} \
    --free_bits {FREE_BITS} \
    --temperature {TEMPERATURE} --clipnorm {CLIPNORM}

## 6. Download the outputs

Zips all run artifacts (weights, config, history, generated SMILES, grids) and triggers a browser download.

In [ ]:
!zip -r -q smiles_vae_outputs.zip outputs/run_*
from google.colab import files
files.download('smiles_vae_outputs.zip')